In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# 1. Load the cleaned dataset
df = pd.read_csv("Group A Deadline-29 October 26 Model_C8 - Cleaned dataset.csv")


# 2. Clean Rating and Review columns
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

df["Review"] = pd.to_numeric(
    df["Review"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

In [3]:
# 3. Identify businesses with and without review data
df["Review_Data_Available"] = np.where(
    (df["Rating"] == 0) & (df["Review"] == 0),
    "No Review Data",
    "Review Data Available"
)


# 4. Calculate Rating Score
df["Rating_Score"] = df["Rating"] / 5


# 5. Calculate Review Volume Score
df["Log_Reviews"] = np.log1p(df["Review"])

scaler = MinMaxScaler()

df["Review_Volume_Score"] = scaler.fit_transform(
    df[["Log_Reviews"]]
)


# 6. Calculate the combined Review Score
df["Review_Score"] = (
    0.60 * df["Rating_Score"] +
    0.40 * df["Review_Volume_Score"]
)


In [4]:


# 7. Classify businesses into Low, Medium, High, and No Review Data
Q1 = df["Review_Score"].quantile(0.25)
Q3 = df["Review_Score"].quantile(0.75)


def classify_review(row):
    if row["Review_Data_Available"] == "No Review Data":
        return "No Review Data"

    elif row["Review_Score"] <= Q1:
        return "Low"

    elif row["Review_Score"] <= Q3:
        return "Medium"

    else:
        return "High"


df["Review_Class"] = df.apply(classify_review, axis=1)

In [5]:
# 8. Display the distribution of review classes
print("Review Class Distribution:")
print(df["Review_Class"].value_counts())


# 9. Prepare the data for model training
model_df = df[
    (df["Review_Data_Available"] == "Review Data Available") &
    df["Rating"].notna() &
    df["Review"].notna()
].copy()


# Features and target
X = model_df[["Rating", "Review"]]
y = model_df["Review_Class"]



Review Class Distribution:
Review_Class
Medium            81
High              42
No Review Data    24
Low               17
Name: count, dtype: int64


In [6]:

# 10. Split data: 60% training and 40% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.60,
    test_size=0.40,
    random_state=42,
    stratify=y
)


In [7]:
print("\nTraining Records:", len(X_train))
print("Testing Records:", len(X_test))


# 11. Train the Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)




Training Records: 83
Testing Records: 56


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
# 12. Evaluate the model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix (Low, Medium, High):")
print(
    confusion_matrix(
        y_test,
        y_pred,
        labels=["Low", "Medium", "High"]
    )
)




Model Accuracy: 0.9286

Classification Report:
              precision    recall  f1-score   support

        High       0.93      0.81      0.87        16
         Low       1.00      1.00      1.00         7
      Medium       0.91      0.97      0.94        33

    accuracy                           0.93        56
   macro avg       0.95      0.93      0.94        56
weighted avg       0.93      0.93      0.93        56


Confusion Matrix (Low, Medium, High):
[[ 7  0  0]
 [ 0 32  1]
 [ 0  3 13]]


In [9]:
# 13. Display feature importance
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(feature_importance)


# 14. Predict review classes for all eligible businesses
df["Predicted_Review_Class"] = "No Review Data"

df["Predicted_Review_Probability"] = np.nan

all_valid = (
    (df["Review_Data_Available"] == "Review Data Available") &
    df["Rating"].notna() &
    df["Review"].notna()
)

df.loc[all_valid, "Predicted_Review_Class"] = model.predict(
    df.loc[all_valid, ["Rating", "Review"]]
)




Feature Importance:
  Feature  Importance
1  Review     0.53793
0  Rating     0.46207


In [10]:
# 15. Calculate prediction probabilities
probabilities = model.predict_proba(
    df.loc[all_valid, ["Rating", "Review"]]
)

df.loc[
    all_valid,
    "Predicted_Review_Probability"
] = probabilities.max(axis=1)


# 16. Export the final results
output_file = "BLC_C8_Review_Analysis_Output.csv"

df.to_csv(output_file, index=False)

print("\nReview analysis completed successfully!")
print("Output saved as:", output_file)


Review analysis completed successfully!
Output saved as: BLC_C8_Review_Analysis_Output.csv


In [11]:
# Create separate datasets for each review class
high_df = df[df["Review_Class"] == "High"]
medium_df = df[df["Review_Class"] == "Medium"]
low_df = df[df["Review_Class"] == "Low"]
no_review_df = df[df["Review_Class"] == "No Review Data"]

In [12]:
# Save each dataset as a separate CSV file
high_df.to_csv("BLC_C8_High_Review.csv", index=False)
medium_df.to_csv("BLC_C8_Medium_Review.csv", index=False)
low_df.to_csv("BLC_C8_Low_Review.csv", index=False)
no_review_df.to_csv("BLC_C8_No_Review_Data.csv", index=False)

In [13]:
# Display the number of records in each file
print("High Review businesses:", len(high_df))
print("Medium Review businesses:", len(medium_df))
print("Low Review businesses:", len(low_df))
print("Businesses with No Review Data:", len(no_review_df))

print("\nAll four CSV files have been created successfully!")

High Review businesses: 42
Medium Review businesses: 81
Low Review businesses: 17
Businesses with No Review Data: 24

All four CSV files have been created successfully!


In [14]:
# Clean the Country column
df["Country"] = df["Country"].astype("string").str.strip()

# Classify businesses based on their country
df["Business_Type"] = df["Country"].apply(
    lambda country: (
        "Indian" if pd.notna(country) and country.casefold() == "india"
        else "Foreign" if pd.notna(country) and country.strip() != ""
        else "Unknown"
    )
)


In [15]:
# Create separate datasets
indian_df = df[df["Business_Type"] == "Indian"]
foreign_df = df[df["Business_Type"] == "Foreign"]
unknown_df = df[df["Business_Type"] == "Unknown"]

# Save the datasets
indian_df.to_csv("BLC_C8_Indian_Businesses.csv", index=False)
foreign_df.to_csv("BLC_C8_Foreign_Businesses.csv", index=False)
unknown_df.to_csv("BLC_C8_Unknown_Country.csv", index=False)

# Display the results
print("Indian businesses:", len(indian_df))
print("Foreign businesses:", len(foreign_df))
print("Businesses with missing country:", len(unknown_df))

print("\nCountry classification completed successfully!")

Indian businesses: 163
Foreign businesses: 1
Businesses with missing country: 0

Country classification completed successfully!


In [16]:
# Check available columns to identify the model's priority column
print(df.columns.tolist())

['S.No', 'Country', 'City', 'State', 'Company', 'CONTACT', 'Google Map', 'Category', 'Clean_Category', 'Rating', 'Review', 'Review_Data_Available', 'Rating_Score', 'Log_Reviews', 'Review_Volume_Score', 'Review_Score', 'Review_Class', 'Predicted_Review_Class', 'Predicted_Review_Probability', 'Business_Type']


In [17]:
df["Clean_Category"].value_counts()

Clean_Category
Restaurant                        19
Apparel Store                     17
General Store                     16
Electronics                       15
Mobile Phone Store                14
Pharmacy                           8
Automotive & Mobility Services     5
General Hardware Store             5
Salon                              5
Jewellers                          4
Building Material Supplier         4
Agricultural Seed Store            4
Clinic                             3
Furniture Store                    3
Photo Studio                       3
Sweet Shop                         2
Coaching Center                    2
Bakery                             2
Electronic Repair Shop             2
Hotel                              2
Stationery                         2
Sports club                        2
Handicrafts                        2
Watch Shop                         2
Bag Supplier                       2
Clothes Manufacturer               2
Paint Dealer           

In [18]:
comparison = pd.crosstab(
    df["Review_Class"],
    df["Predicted_Review_Class"],
    dropna=False
)

comparison

Predicted_Review_Class,High,Low,Medium,No Review Data
Review_Class,,,,
High,38,0,3,1
Low,0,17,0,0
Medium,1,0,80,0
No Review Data,0,0,0,24


In [19]:
print(df.columns.tolist())

['S.No', 'Country', 'City', 'State', 'Company', 'CONTACT', 'Google Map', 'Category', 'Clean_Category', 'Rating', 'Review', 'Review_Data_Available', 'Rating_Score', 'Log_Reviews', 'Review_Volume_Score', 'Review_Score', 'Review_Class', 'Predicted_Review_Class', 'Predicted_Review_Probability', 'Business_Type']


In [25]:
import pandas as pd

# Load the complete BLC-C8 model output
df = pd.read_csv(r"C:\Users\arshi\OneDrive\Desktop\BLC-C8 PROJECT\BLC_C8_Review_Analysis_Output.csv")

# Clean category names
df["Clean_Category"] = df["Clean_Category"].astype("string").str.strip()

# Count predicted classifications for each category
counts = pd.crosstab(
    df["Clean_Category"],
    df["Predicted_Review_Class"]
)

classes = ["High", "Medium", "Low", "No Review Data"]

# Ensure all classes exist
for c in classes:
    if c not in counts.columns:
        counts[c] = 0

counts = counts[classes]

# Assign each category its most frequent predicted class
# Tie-break order: High > Medium > Low > No Review Data
category_class = counts.idxmax(axis=1)

# Generate unique category lists
HIGH = sorted(category_class[category_class == "High"].index.tolist())
MEDIUM = sorted(category_class[category_class == "Medium"].index.tolist())
LOW = sorted(category_class[category_class == "Low"].index.tolist())
NO_REVIEW_DATA = sorted(
    category_class[category_class == "No Review Data"].index.tolist()
)

# Print the requested format
print("HIGH =", HIGH)
print("MEDIUM =", MEDIUM)
print("LOW =", LOW)
print("NO_REVIEW_DATA =", NO_REVIEW_DATA)

# Verify that every category appears exactly once
all_categories = HIGH + MEDIUM + LOW + NO_REVIEW_DATA

print("\nTotal unique categories:", len(all_categories))
print("All categories unique:", len(all_categories) == len(set(all_categories)))

HIGH = ['Bakery', 'Clinic', 'Dairy', 'Electronic Repair Shop', 'Furniture Store', 'Handicrafts', 'Hotel', 'Laundry Service', 'Retail Store', 'Salon', 'Sports club', 'Sweet Shop', 'Tile and Stone Showroom', 'Tobacco shop', 'Watch Shop']
MEDIUM = ['Agricultural Seed Store', 'Apparel Store', 'Automotive & Mobility Services', 'Bag Supplier', 'Building Material Supplier', 'Clothes Manufacturer', 'Coaching Center', 'Computer Store', 'Electronics', 'General Hardware Store', 'General Store', 'Gift Store', 'Gym Supplement Store', 'Hostel', 'Jewellers', 'Mobile Phone Store', 'Musical Instrument Store', 'Pharmacy', 'Photo Studio', 'Printing Service', 'Restaurant', 'Tailor']
LOW = ['Animal Feed Store', 'Confectionary Store', 'Flower Shop', 'Stationery']
NO_REVIEW_DATA = ['Manufacturing', 'Paint Dealer']

Total unique categories: 43
All categories unique: True


In [26]:

# Ensure Rating and Review are numeric
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce").fillna(0)
df["Review"] = pd.to_numeric(
    df["Review"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
).fillna(0)

# Identify businesses with no review data
no_data = (df["Rating"] == 0) & (df["Review"] == 0)

# Calculate thresholds using businesses that have review data
scored = df.loc[~no_data, "Review_Score"].dropna()

Q1 = scored.quantile(0.25)
Q3 = scored.quantile(0.75)

print(f"High Priority: Score > {Q3:.6f}")
print(f"Medium Priority: Score > {Q1:.6f} and <= {Q3:.6f}")
print(f"Low Priority: Score <= {Q1:.6f}")
print("No Review Data: Both Rating and Review count are 0")

High Priority: Score > 0.710346
Medium Priority: Score > 0.628025 and <= 0.710346
Low Priority: Score <= 0.628025
No Review Data: Both Rating and Review count are 0


In [27]:
import pandas as pd

# Work with your existing BLC-C8 model output DataFrame: df
data = df.copy()

# Clean category names
data["Clean_Category"] = data["Clean_Category"].astype("string").str.strip()

# Convert rating and review count to numeric
data["Rating"] = pd.to_numeric(data["Rating"], errors="coerce").fillna(0)

data["Review"] = pd.to_numeric(
    data["Review"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
).fillna(0)

# Identify businesses with no review data
no_data = (data["Rating"] == 0) & (data["Review"] == 0)

# Separate businesses with review data from those without it
reviewed = data.loc[~no_data].copy()
unreviewed = data.loc[no_data].copy()

# Normalize predicted class labels
reviewed["Predicted_Review_Class"] = (
    reviewed["Predicted_Review_Class"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Count each model-predicted class within each category
counts = pd.crosstab(
    reviewed["Clean_Category"],
    reviewed["Predicted_Review_Class"]
)

# Ensure all three classes are represented
for cls in ["High", "Medium", "Low"]:
    if cls not in counts.columns:
        counts[cls] = 0

counts = counts[["High", "Medium", "Low"]]

# Assign each category its most frequent predicted class.
# Ties are resolved in this order: High, Medium, Low.
category_class = counts.idxmax(axis=1)

# Categories whose businesses have no review data at all
no_review_categories = set(unreviewed["Clean_Category"].dropna()) - set(
    reviewed["Clean_Category"].dropna()
)

# Create unique category lists
HIGH = sorted(
    category_class[category_class == "High"].index.difference(
        no_review_categories
    ).tolist()
)

MEDIUM = sorted(
    category_class[category_class == "Medium"].index.difference(
        no_review_categories
    ).tolist()
)

LOW = sorted(
    category_class[category_class == "Low"].index.difference(
        no_review_categories
    ).tolist()
)

NO_REVIEW_DATA = sorted(no_review_categories)

# Print results in the requested format
print("HIGH =", HIGH)
print("MEDIUM =", MEDIUM)
print("LOW =", LOW)
print("NO_REVIEW_DATA =", NO_REVIEW_DATA)

# Verify that every category appears only once
all_categories = HIGH + MEDIUM + LOW + NO_REVIEW_DATA

print("\nTotal unique categories:", len(all_categories))
print("No duplicate categories:", len(all_categories) == len(set(all_categories)))

HIGH = ['Bakery', 'Clinic', 'Dairy', 'Electronic Repair Shop', 'Furniture Store', 'Handicrafts', 'Hotel', 'Laundry Service', 'Retail Store', 'Salon', 'Sports club', 'Sweet Shop', 'Tile and Stone Showroom', 'Tobacco shop', 'Watch Shop']
MEDIUM = ['Agricultural Seed Store', 'Apparel Store', 'Automotive & Mobility Services', 'Bag Supplier', 'Building Material Supplier', 'Clothes Manufacturer', 'Coaching Center', 'Computer Store', 'Electronics', 'General Hardware Store', 'General Store', 'Gift Store', 'Gym Supplement Store', 'Hostel', 'Jewellers', 'Mobile Phone Store', 'Musical Instrument Store', 'Pharmacy', 'Photo Studio', 'Printing Service', 'Restaurant', 'Tailor']
LOW = ['Animal Feed Store', 'Confectionary Store', 'Flower Shop', 'Stationery']
NO_REVIEW_DATA = ['Manufacturing', 'Paint Dealer']

Total unique categories: 43
No duplicate categories: True


In [28]:
print("Review class distribution:")
print(df["Review_Class"].value_counts(dropna=False))

print("\nPredicted review class distribution:")
print(df["Predicted_Review_Class"].value_counts(dropna=False))

print("\nReview score summary:")
print(df["Review_Score"].describe())

print("\nSample records:")
print(
    df[[
        "Clean_Category",
        "Rating",
        "Review",
        "Review_Score",
        "Review_Class",
        "Predicted_Review_Class"
    ]].head(10).to_string(index=False)
)

Review class distribution:
Review_Class
Medium            81
High              42
No Review Data    24
Low               17
Name: count, dtype: int64

Predicted review class distribution:
Predicted_Review_Class
Medium            83
High              39
No Review Data    25
Low               17
Name: count, dtype: int64

Review score summary:
count    163.000000
mean       0.565466
std        0.253845
min        0.000000
25%        0.579116
50%        0.650154
75%        0.696974
max        0.940000
Name: Review_Score, dtype: float64

Sample records:
            Clean_Category  Rating  Review  Review_Score   Review_Class Predicted_Review_Class
                     Salon     5.0    20.0      0.738990           High                   High
              Paint Dealer     0.0     0.0      0.000000 No Review Data         No Review Data
                    Clinic     5.0     1.0      0.631644         Medium                 Medium
Building Material Supplier     4.3     6.0      0.604835        